# 第二章（二）：多声部与符号数据清洗

以 J. S. Bach《平均律钢琴曲集》第一卷 BWV 854 赋格为例，区分 SMF 轨道、MIDI 通道、`pretty_midi.Instrument`、MusicXML `part`、谱表和逻辑声部，并记录第1—17小节的跨格式对齐。

数据：
- MIDI: `CODE/datasets/lmd_clean_midi/Bach Johann Sebastian/Bach Prelude and Fugue in E major BWV 854 Fugue.mid`
- MusicXML: `CODE/datasets/MusicXML/Bach Prelude and Fugue in E major BWV 854 Fugue.musicxml`
- 校验与对齐元数据文件（sidecar）：`BWV854_fugue_sidecar.yaml`

图片输出：`CODE/chapter02/output_figures/`（600 dpi）

In [ ]:
import hashlib
import os
import warnings
from collections import Counter
from xml.etree import ElementTree as ET

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import mido
import music21
import numpy as np
import pretty_midi
import yaml

warnings.filterwarnings("ignore", message="Tempo, Key or Time signature change events found.*", category=RuntimeWarning)

# 中文字体
plt.rcParams["font.sans-serif"] = [
    "PingFang SC", "Hiragino Sans GB", "Microsoft YaHei",
    "SimHei", "Arial Unicode MS", "Noto Sans CJK SC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

# 强制白底
for _k in ("figure.facecolor", "axes.facecolor"):
    plt.rcParams[_k] = "white"
for _k in ("axes.edgecolor", "axes.labelcolor", "xtick.color", "ytick.color", "text.color"):
    plt.rcParams[_k] = "black"

# 路径
_p = os.getcwd()
while not os.path.exists(os.path.join(_p, "CODE", "datasets")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError("未找到包含 CODE/datasets 的项目根目录")
    _p = _parent
BASE_DIR = _p

BACH_DIR = os.path.join(
    BASE_DIR, "CODE", "datasets", "lmd_clean_midi", "Bach Johann Sebastian"
)
MIDI_PATH = os.path.join(
    BACH_DIR, "Bach Prelude and Fugue in E major BWV 854 Fugue.mid"
)
SIDECAR_PATH = os.path.join(BACH_DIR, "BWV854_fugue_sidecar.yaml")
MUSICXML_PATH = os.path.join(
    BASE_DIR, "CODE", "datasets", "MusicXML",
    "Bach Prelude and Fugue in E major BWV 854 Fugue.musicxml"
)
FIGURES_DIR = os.path.join(BASE_DIR, "CODE", "chapter02", "output_figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

print("MIDI_PATH：", MIDI_PATH)
print("MUSICXML_PATH：", MUSICXML_PATH)
print("SIDECAR_PATH：", SIDECAR_PATH)

## 1. 加载 BWV 854 标准 MIDI 文件

In [ ]:
smf = mido.MidiFile(MIDI_PATH)
pm = pretty_midi.PrettyMIDI(MIDI_PATH)

print(f"SMF 类型：{smf.type}；每四分音符 tick 数：{smf.ticks_per_beat}")
print(f"SMF 轨道数：{len(smf.tracks)}")
for track_index, track in enumerate(smf.tracks):
    note_on_count = sum(1 for message in track if message.type == "note_on" and message.velocity > 0)
    print(f"  轨道 {track_index}：name={track.name!r}，起音消息数={note_on_count}")

print(f"pretty_midi Instrument 对象数：{len(pm.instruments)}")
for instrument_index, instrument in enumerate(pm.instruments):
    print(f"  Instrument {instrument_index}：program={instrument.program}，"
          f"name={instrument.name!r}，is_drum={instrument.is_drum}，Note 对象数={len(instrument.notes)}")

all_notes = []
for inst in pm.instruments:
    if not inst.is_drum:
        all_notes.extend(inst.notes)
all_notes.sort(key=lambda note: (note.start, note.pitch, note.end))

print(f"\nNote 对象总数：{len(all_notes)}")
print(f"音高范围：MIDI {min(n.pitch for n in all_notes)} "
      f"({pretty_midi.note_number_to_name(min(n.pitch for n in all_notes))})～"
      f"{max(n.pitch for n in all_notes)} "
      f"({pretty_midi.note_number_to_name(max(n.pitch for n in all_notes))})")
print(f"文件终止时间：{pm.get_end_time():.2f} 秒")

## 2. SMF 元事件与轨道名称

速度、拍号和轨道名位于 SMF 元事件中，不属于只包含格式类型、轨道数和时间分辨率的文件头。代码读取这些字段，并与配套 MusicXML 分开记录。

In [ ]:
tempo_times, tempos = pm.get_tempo_changes()
time_sigs = pm.time_signature_changes
key_sigs = pm.key_signature_changes

print("=== SMF 元事件与轨道名称 ===")
print(f"速度元事件：{tempos} BPM")
print(f"拍号元事件：{time_sigs}")
print(f"调号元事件：{key_sigs if key_sigs else '(无)'}")
print(f"轨道名称：{[track.name for track in smf.tracks]}")
print(f"说明：{tempos[0]:.0f} BPM 是文件实际采用的速度元事件；配套 MusicXML 另有四分音符=175 的记谱标记。")
print(f"说明：{time_sigs[0].numerator}/{time_sigs[0].denominator} 元事件与配套 MusicXML 的 3/4 记谱不一致，不能用于本章的小节切分。")

## 3. 校验与对齐 sidecar 文件

伴随元数据文件（sidecar）记录源文件哈希、观测字段、跨格式对齐方法、适用范围和未确认事项。该文件不修改原始 MIDI，也不把来源不明的 MusicXML 标记为权威校勘谱。

In [ ]:
with open(SIDECAR_PATH, "r", encoding="utf-8") as f:
    sidecar = yaml.safe_load(f)

def sha256(path):
    with open(path, "rb") as file_object:
        return hashlib.sha256(file_object.read()).hexdigest()

assert sha256(MIDI_PATH) == sidecar["source_files"]["midi"]["sha256"]
assert sha256(MUSICXML_PATH) == sidecar["source_files"]["musicxml"]["sha256"]
print("源文件 SHA-256 与 sidecar 一致。")

# 程序核对第2节两条说明所依据的速度与拍号
smf_observed = sidecar["smf_observed"]
musicxml_observed = sidecar["musicxml_observed"]
assert len(tempos) == 1 and len(time_sigs) == 1
runtime_tempo_bpm = float(tempos[0])
runtime_meter = f"{time_sigs[0].numerator}/{time_sigs[0].denominator}"
assert runtime_tempo_bpm == float(smf_observed["tempo_meta_bpm"])
assert runtime_meter == smf_observed["time_signature_meta"]
assert float(musicxml_observed["tempo_mark_bpm"]) != runtime_tempo_bpm
assert musicxml_observed["notated_meter"] != runtime_meter
print(f"SMF 速度/拍号运行观测与 sidecar 一致：{runtime_tempo_bpm:.0f} BPM、{runtime_meter}；"
      f"与 MusicXML 记谱 {musicxml_observed['tempo_mark_bpm']} BPM、{musicxml_observed['notated_meter']} 不一致，"
      f"第2节两条说明核对成立。")
print("作品：", sidecar["work"])
print("SMF 观测：", sidecar["smf_observed"])
print("MusicXML 观测：", sidecar["musicxml_observed"])
print("限制：")
for limitation in sidecar["limitations"]:
    print(" -", limitation)

In [ ]:
# SMF 与配套 MusicXML 对照
print(f"{'字段':<18} {'SMF':<28} {'配套 MusicXML / 分析':<36}")
print("-" * 86)
rows = [
    ("速度", f"{tempos[0]:.0f} BPM", f"记谱标记 {sidecar['musicxml_observed']['tempo_mark_bpm']} BPM；切分使用对齐"),
    ("拍号", f"{runtime_meter} 元事件", sidecar["musicxml_observed"]["notated_meter"]),
    ("调号", str(key_sigs) if key_sigs else "未设置", sidecar["musicxml_observed"]["interpreted_key"]),
    ("轨道/part", f"{len(smf.tracks)} 个轨道", f"{sidecar['musicxml_observed']['part_count']} 个 part，{sidecar['musicxml_observed']['staff_count']} 个谱表"),
    ("首个起音", f"{all_notes[0].start:.2f} 秒", "分析图中设为相对 0 秒"),
]
for field, midi_val, sidecar_val in rows:
    print(f"{field:<18} {midi_val:<28} {sidecar_val:<36}")

print("原始 SMF 保持不变；分析使用单独记录的小节对齐，不改写速度元事件。")

## 4. 截取至第17小节

第1—17小节的边界来自 MusicXML—MIDI 起音事件匹配。程序按出现顺序配对两种文件中相同音高的第 n 次起音，因此假设相同音高的起音顺序保持一致。该方法不能稳健处理漏音、多音或相同音高事件重排；遇到这些情况，需要显式序列对齐与人工复核。第18小节首个起音相对于文件起点的时间为 19.4322917 秒，片段边界设在该时刻。SMF 的 120 BPM 速度元事件保持不变；MusicXML 的四分音符=175 记谱标记不用于重缩放 MIDI 时间。


In [ ]:
alignment = sidecar["alignment"]
excerpt_start_measure = int(alignment["excerpt_start_measure"])
excerpt_end_measure = int(alignment["excerpt_end_measure"])
excerpt_measures = excerpt_end_measure - excerpt_start_measure + 1
recorded_measure_starts = {
    int(item["measure"]): float(item["start"])
    for item in alignment["measure_starts_seconds"]
}

step_to_pitch_class = {"C": 0, "D": 2, "E": 4, "F": 5, "G": 7, "A": 9, "B": 11}

def extract_musicxml_onsets_with_score_time(path):
    """提取有音高且不是延音线续接音的起音，并保留乐谱时间。"""
    root = ET.parse(path).getroot()
    parts = root.findall("part")
    if len(parts) != 1:
        raise ValueError("本例的对齐代码要求 MusicXML 只有一个 part")

    divisions = 1
    measure_start_quarters = 0.0
    measure_score_starts = {}
    onset_records = []
    for measure in parts[0].findall("measure"):
        measure_number = int(measure.attrib["number"])
        measure_score_starts[measure_number] = measure_start_quarters
        current_position = 0.0
        maximum_position = 0.0
        previous_note_onset = 0.0

        for child in measure:
            if child.tag == "attributes":
                divisions_text = child.findtext("divisions")
                if divisions_text is not None:
                    divisions = int(divisions_text)
            elif child.tag == "note":
                duration = int(child.findtext("duration", default="0")) / divisions
                is_chord_tone = child.find("chord") is not None
                note_onset = previous_note_onset if is_chord_tone else current_position
                if not is_chord_tone:
                    previous_note_onset = note_onset

                pitch = child.find("pitch")
                tie_types = {tie.attrib.get("type") for tie in child.findall("tie")}
                if pitch is not None and "stop" not in tie_types:
                    step = pitch.findtext("step")
                    alter = int(pitch.findtext("alter", default="0"))
                    octave = int(pitch.findtext("octave"))
                    onset_records.append({
                        "measure": measure_number,
                        "score_time_quarters": measure_start_quarters + note_onset,
                        "pitch": 12 * (octave + 1) + step_to_pitch_class[step] + alter,
                    })

                if not is_chord_tone:
                    current_position += duration
                    maximum_position = max(maximum_position, current_position)
            elif child.tag == "backup":
                current_position -= int(child.findtext("duration")) / divisions
            elif child.tag == "forward":
                current_position += int(child.findtext("duration")) / divisions
                maximum_position = max(maximum_position, current_position)

        if maximum_position <= 0:
            raise ValueError(f"第{measure_number}小节没有可用的时间跨度")
        measure_start_quarters += maximum_position
    return measure_score_starts, onset_records

xml_measure_score_starts, xml_onset_records = extract_musicxml_onsets_with_score_time(MUSICXML_PATH)
xml_onset_records.sort(key=lambda record: (record["score_time_quarters"], record["pitch"]))
midi_starts_by_pitch = {
    pitch: [note.start for note in all_notes if note.pitch == pitch]
    for pitch in {note.pitch for note in all_notes}
}
# 按同音高的出现序号匹配；前提是两种文件保持同音高起音顺序。
# 若有漏音、多音或重复音重排，应改用显式序列对齐并人工复核。
pitch_occurrences = Counter()
for onset_record in xml_onset_records:
    pitch = onset_record["pitch"]
    occurrence_index = pitch_occurrences[pitch]
    if occurrence_index >= len(midi_starts_by_pitch.get(pitch, [])):
        raise ValueError(f"MusicXML 音高 {pitch} 的起音数多于 MIDI")
    onset_record["midi_time"] = midi_starts_by_pitch[pitch][occurrence_index]
    pitch_occurrences[pitch] += 1
assert all(
    pitch_occurrences[pitch] == len(starts)
    for pitch, starts in midi_starts_by_pitch.items()
)

reconstructed_measure_starts = {}
score_time_tolerance = 1e-12
for measure_number in range(excerpt_start_measure, excerpt_end_measure + 2):
    score_start = xml_measure_score_starts[measure_number]
    onsets_at_start = [
        record["midi_time"]
        for record in xml_onset_records
        if np.isclose(record["score_time_quarters"], score_start,
                      atol=score_time_tolerance, rtol=0)
    ]
    if onsets_at_start:
        reconstructed_measure_starts[measure_number] = min(onsets_at_start)
    else:
        before = max(
            (record for record in xml_onset_records
             if record["score_time_quarters"] < score_start),
            key=lambda record: record["score_time_quarters"],
        )
        after = min(
            (record for record in xml_onset_records
             if record["score_time_quarters"] > score_start),
            key=lambda record: record["score_time_quarters"],
        )
        interpolation_ratio = (
            (score_start - before["score_time_quarters"])
            / (after["score_time_quarters"] - before["score_time_quarters"])
        )
        reconstructed_measure_starts[measure_number] = (
            before["midi_time"]
            + interpolation_ratio * (after["midi_time"] - before["midi_time"])
        )

assert all(
    np.isclose(reconstructed_measure_starts[measure], recorded_measure_starts[measure],
               atol=1e-9, rtol=0)
    for measure in reconstructed_measure_starts
)
measure_starts = recorded_measure_starts
print("第1—18小节起点已从 MusicXML—MIDI 起音匹配中重建，并与 sidecar 一致。")
first_onset = measure_starts[excerpt_start_measure]
excerpt_end = measure_starts[excerpt_end_measure + 1]
boundary_note_time = min(
    note.start for note in all_notes
    if np.isclose(note.start, excerpt_end, atol=1e-9, rtol=0)
)
excerpt_duration = excerpt_end - first_onset
relative_measure_starts = [
    measure_starts[measure] - first_onset
    for measure in range(excerpt_start_measure, excerpt_end_measure + 2)
]

excerpt_notes = [n for n in all_notes if first_onset <= n.start < boundary_note_time]
assert len(excerpt_notes) == int(alignment["excerpt_note_on_count"])
print(f"截取范围：第{excerpt_start_measure}—{excerpt_end_measure}小节")
print(f"起点：{first_onset:.10f} 秒")
print(f"终点记录值：{excerpt_end:.10f} 秒")
print(f"MIDI 边界起音：{boundary_note_time:.16f} 秒（相对于文件起点，第{excerpt_end_measure + 1}小节）")
print(f"片段时长：{excerpt_duration:.10f} 秒")
print(f"起音事件数：{len(excerpt_notes)} / 全曲 {len(all_notes)}")


## 5. 多声部钢琴卷帘

钢琴卷帘按 0.01 秒采样，并以 CC64 阈值 64 计入踏板延音。图中的纵向重叠表示同一采样时刻存在多个活动音高，不等同于赋格的逻辑声部数。

In [ ]:
sample_step = float(sidecar["density_analysis"]["sample_step_seconds"])
sustain_threshold = int(sidecar["density_analysis"]["sustain_threshold"])
sample_times = np.arange(first_onset, excerpt_end, sample_step)
active_matrix = np.zeros((128, len(sample_times)), dtype=bool)

def sounding_end_time(note_end, sustain_changes, threshold, file_end):
    """返回计入延音踏板后的终止时间。"""
    pedal_is_down = False
    for control_change in sustain_changes:
        if control_change.time > note_end:
            break
        pedal_is_down = control_change.value >= threshold
    if not pedal_is_down:
        return note_end
    return next(
        (control_change.time for control_change in sustain_changes
         if control_change.time > note_end and control_change.value < threshold),
        file_end,
    )

for roll_instrument in pm.instruments:
    if roll_instrument.is_drum:
        continue
    if roll_instrument.pitch_bends:
        raise ValueError("当前活动音高统计未处理弯音消息")
    sustain_changes = sorted(
        (cc for cc in roll_instrument.control_changes if cc.number == 64),
        key=lambda cc: cc.time,
    )
    for roll_note in roll_instrument.notes:
        effective_end = min(
            sounding_end_time(roll_note.end, sustain_changes, sustain_threshold, pm.get_end_time()),
            excerpt_end,
        )
        if roll_note.start < excerpt_end and effective_end > first_onset:
            active_matrix[roll_note.pitch] |= (
                (sample_times >= roll_note.start) & (sample_times < effective_end)
            )
time_grid = sample_times - first_onset

pitches = [note.pitch for note in excerpt_notes]
pitch_min = min(pitches)
pitch_max = max(pitches)
figure_title = (
    f"BWV 854 赋格第{excerpt_start_measure}—{excerpt_end_measure}小节钢琴卷帘"
    "（计入 CC64 延音）"
)
print(f"图：{figure_title}")
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(
    active_matrix[pitch_min:pitch_max + 1],
    origin="lower",
    aspect="auto",
    interpolation="nearest",
    cmap=mcolors.ListedColormap(["white", "#2c3e50"]),
    extent=[0, excerpt_duration, pitch_min - 0.5, pitch_max + 0.5],
)

# 小节线
for boundary_index, boundary_time in enumerate(relative_measure_starts):
    t = boundary_time
    ax.axvline(t, color="gray", linestyle="--", linewidth=0.6, alpha=0.5)
    if boundary_index < excerpt_measures:
        next_time = relative_measure_starts[boundary_index + 1]
        ax.text((t + next_time) / 2, pitch_max + 1.2,
                str(excerpt_start_measure + boundary_index), ha="center", fontsize=7, color="gray")

ax.set_xlim(0, excerpt_duration)
ax.set_ylim(pitch_min - 1, pitch_max + 2)
yticks = list(range(pitch_min, pitch_max + 1))
ax.set_yticks(yticks)
ax.set_yticklabels([pretty_midi.note_number_to_name(p) for p in yticks], fontsize=7)

ax.set_xlabel("时间（秒，相对于第1小节起点）")
ax.set_ylabel("MIDI 音高")
ax.grid(axis="y", linestyle=":", alpha=0.2)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, "piano_roll_bwv854.png")
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"已保存：{out_path}")

## 6. 统计分析

起音统计以第1—17小节内的 154 个 `pretty_midi.Note` 对象为单位。各音高类按 MIDI 音符编号对 12 取模统计，等音拼写合并，图中统一使用升号音名。对象时长在第18小节边界处裁切；活动音高数按 0.01 秒采样时刻统计，并计入 CC64 延音。

In [ ]:
onset_pitches = [note.pitch for note in excerpt_notes]
excerpt_note_durations = [min(note.end, boundary_note_time) - note.start for note in excerpt_notes]
clipped_note_count = sum(note.end > boundary_note_time for note in excerpt_notes)
assert all(duration > 0 for duration in excerpt_note_durations)

print("=== 起音事件统计 ===")
print(f"起音事件数：{len(onset_pitches)}")
print(f"音高范围：{min(onset_pitches)}（{pretty_midi.note_number_to_name(min(onset_pitches))}）～"
      f"{max(onset_pitches)}（{pretty_midi.note_number_to_name(max(onset_pitches))}）")
print(f"音域跨度：{max(onset_pitches) - min(onset_pitches)} 半音")
print(f"片段内裁切后 Note 时长均值：{np.mean(excerpt_note_durations):.3f} 秒")
print(f"片段内裁切后 Note 时长范围：{min(excerpt_note_durations):.3f}～{max(excerpt_note_durations):.3f} 秒")
print(f"跨越第{excerpt_end_measure + 1}小节边界并裁切的 Note 对象数：{clipped_note_count}")
pitch_class_counts = Counter(pitch % 12 for pitch in onset_pitches)
print(f"B 起音次数：{pitch_class_counts[11]}；E 起音次数：{pitch_class_counts[4]}")

In [ ]:
active_pitch_counts = np.count_nonzero(active_matrix, axis=0)
mean_active_pitches = float(active_pitch_counts.mean())
max_active_pitches = int(active_pitch_counts.max())
fraction_over_two_percent = float((active_pitch_counts > 2).mean() * 100)

expected_density = sidecar["density_analysis"]
assert np.isclose(mean_active_pitches, expected_density["mean_active_pitches"])
assert max_active_pitches == int(expected_density["max_active_pitches"])
assert np.isclose(fraction_over_two_percent, expected_density["fraction_over_two_percent"])

print("=== 活动音高数（0.01 秒采样，计入 CC64 延音） ===")
print(f"平均活动音高数：{mean_active_pitches:.4f}")
print(f"最大活动音高数：{max_active_pitches}")
print(f"活动音高数大于 2 的时间占比：{fraction_over_two_percent:.1f}%")

In [ ]:
figure_title = (
    f"BWV 854 赋格第{excerpt_start_measure}—{excerpt_end_measure}小节的"
    "各音高类起音次数与活动音高数（计入 CC64 延音）"
)
print(f"图：{figure_title}")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 音高分布
ax = axes[0]
pitch_classes = [pitch % 12 for pitch in onset_pitches]
labels = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]
counts = [pitch_classes.count(pc) for pc in range(12)]
colors = ["#2c3e50" if c > 0 else "#ecf0f1" for c in counts]
ax.bar(labels, counts, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("音高类")
ax.set_ylabel("起音次数")

# 活动音高数随时间变化
ax = axes[1]
ax.fill_between(time_grid, active_pitch_counts, alpha=0.4, color="#2c3e50")
ax.plot(time_grid, active_pitch_counts, color="#2c3e50", linewidth=0.5)
for boundary_time in relative_measure_starts:
    ax.axvline(boundary_time, color="gray", linestyle="--", linewidth=0.4, alpha=0.4)
ax.set_xlabel("时间（秒，相对于第1小节起点）")
ax.set_ylabel("活动音高数")
ax.set_xlim(0, excerpt_duration)

plt.tight_layout()
out_path = os.path.join(FIGURES_DIR, "bwv854_statistics.png")
fig.savefig(out_path, dpi=600, bbox_inches="tight")
plt.show()
print(f"已保存：{out_path}")

## 7. 乐谱结构层：MusicXML 对照

代码直接读取原始 XML 层级，并用 `music21` 验证文件可解析。原文件含 1 个 `part`、2 个谱表和 4 个 `voice` 编号。`music21` 为多谱表钢琴谱创建的 `PartStaff` 对象不应被解释为原始 MusicXML 的多个 `part` 或赋格声部。


In [ ]:
score = music21.converter.parse(MUSICXML_PATH)
xml_root = ET.parse(MUSICXML_PATH).getroot()
xml_parts = xml_root.findall("part")
xml_measures = xml_parts[0].findall("measure")
declared_staff_count = int(xml_measures[0].findtext("attributes/staves", default="1"))
voice_ids = sorted({
    int(voice.text)
    for voice in xml_parts[0].iter("voice")
    if voice.text is not None
})

print("=== MusicXML 结构信息 ===")
print(f"原始 part 数：{len(xml_parts)}")
print(f"小节数：{len(xml_measures)}")
print(f"谱表数：{declared_staff_count}")
print(f"voice 编号：{voice_ids}")
print(f"music21 解析后的 PartStaff 数：{len(score.parts)}")
print(f"拍号：{sidecar['musicxml_observed']['notated_meter']}")
print(f"调号：{sidecar['musicxml_observed']['key_signature_fifths']} 个升号，"
      f"mode={sidecar['musicxml_observed']['key_mode']}（大调）")
assert len(xml_parts) == sidecar["musicxml_observed"]["part_count"]
assert declared_staff_count == sidecar["musicxml_observed"]["staff_count"]
assert voice_ids == sidecar["musicxml_observed"]["voice_ids"]


In [ ]:
# SMF 与 MusicXML 信息对照
print(f"{'字段':<16} {'MIDI':<24} {'MusicXML':<24}")
print("-" * 64)
rows = [
    ("拍号", "4/4 元事件", "3/4 记谱"),
    ("调号", "未设置调号元事件", f"4 个升号，{sidecar['musicxml_observed']['interpreted_key']}"),
    ("容器", f"{len(smf.tracks)} 个轨道；{len(pm.instruments)} 个 Instrument",
             f"{len(xml_parts)} 个 part；{declared_staff_count} 个谱表"),
    ("逻辑声部", "未显式记录记谱 voice", f"voice 编号 {voice_ids}"),
    ("小节结构", "本文件元事件不足以可靠切分", f"{len(xml_measures)} 个 measure"),
]
for field, midi_val, xml_val in rows:
    print(f"{field:<16} {midi_val:<24} {xml_val:<24}")

print()
print("两种文件的结构层级不同；轨道、Instrument、part、谱表和 voice 不能相互等同。")
print("该 MusicXML 的原始谱本来源未记录，结构字段仍需按任务范围核验。")

In [ ]:
# 使用前面已提取并参与对齐的 MusicXML 起音记录。
xml_onset_pitches_by_measure = {}
for onset_record in xml_onset_records:
    xml_onset_pitches_by_measure.setdefault(onset_record["measure"], []).append(
        onset_record["pitch"]
    )
xml_full_onset_pitches = [
    pitch
    for measure_number in sorted(xml_onset_pitches_by_measure)
    for pitch in xml_onset_pitches_by_measure[measure_number]
]
xml_excerpt_onset_pitches = [
    pitch
    for measure_number in range(excerpt_start_measure, excerpt_end_measure + 1)
    for pitch in xml_onset_pitches_by_measure[measure_number]
]
print(f"MusicXML 全曲起音事件数：{len(xml_full_onset_pitches)}")
print(f"MIDI 全曲 Note 对象数：{len(all_notes)}")
print(f"MusicXML 第1—{excerpt_end_measure}小节起音事件数：{len(xml_excerpt_onset_pitches)}")
print(f"MIDI 对齐片段起音事件数：{len(excerpt_notes)}")
assert Counter(xml_full_onset_pitches) == Counter(note.pitch for note in all_notes)
assert Counter(xml_excerpt_onset_pitches) == Counter(onset_pitches)
alignment_tolerance = 1e-9
for measure_number in range(excerpt_start_measure, excerpt_end_measure + 1):
    midi_measure_pitches = [
        note.pitch
        for note in all_notes
        if measure_starts[measure_number] - alignment_tolerance <= note.start
        < measure_starts[measure_number + 1] - alignment_tolerance
    ]
    assert Counter(midi_measure_pitches) == Counter(xml_onset_pitches_by_measure[measure_number])
assert alignment["excerpt_measure_pitch_counts_match"] is True
print("全曲起音音高计数一致；第1—17小节的逐小节计数也一致。")

## 8. 符号数据清洗检查单

使用来源复杂的 MIDI 或 MusicXML 数据前，应按任务记录以下检查项：

| 检查项 | 说明 | BWV 854 示例 |
|---|---|---|
| 曲名、作品号与来源 | 文件名和内嵌元数据可能缺失或不一致 | 作品信息来自文件名与配套记录；上游标识和原始制作者未记录 |
| 拍号与小节边界 | 元事件可能缺失或与记谱不一致 | SMF 为 4/4 元事件，MusicXML 记谱为 3/4；第1—17小节使用跨格式对齐 |
| 速度 | 区分播放时间与记谱速度标记 | SMF 使用 120 BPM；MusicXML 标记四分音符=175，不重缩放 MIDI 时间 |
| 调号 | 调号元事件可能缺失 | SMF 未设置；MusicXML 为 4 个升号并标记 `major`（大调） |
| 容器与声部 | 轨道、通道、Instrument、part、谱表和 voice 不能等同 | 3 个 SMF 轨道、1 个 Instrument、1 个 part、2 个谱表、voice 1/2/5/6 |
| 延音线 | 续接音在起音统计中不重复计数 | 排除含 `<tie type="stop">` 的音后，全曲为 735 个起音事件 |
| 统计定义 | 明确时间分辨率、踏板和计数单位 | 0.01 秒采样，CC64 阈值 64，统计活动音高数 |
| 重复与版本 | 保存哈希、来源和划分规则 | sidecar 记录本地文件 SHA-256；原始谱本来源仍未确认 |

| 输出文件 | 内容 |
|---|---|
| `piano_roll_bwv854.png` | BWV 854 第1—17小节钢琴卷帘（计入 CC64 延音） |
| `bwv854_statistics.png` | 各音高类起音次数与活动音高数 |
| `BWV854_fugue_sidecar.yaml` | 来源、观测字段、对齐方法、统计定义与限制 |